[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E4%BA%94%E7%A8%AE%E9%9B%A3%E6%90%9E%E7%9A%84%E8%B3%87%E6%96%99%E6%AA%94.ipynb)

# 📥 茶語時光：搞定五種難搞的資料檔

> 🧋 **情境**：月初結帳日，各部門丟來五種格式的檔案：POS 系統匯出的「髒」CSV、每家店一個工作表的 Excel、外送平台的巢狀 JSON、IT 部門的會員資料庫，還有一個 10 萬筆、用 Excel 打開會卡住的交易檔。你要用 pandas 一一讀進來，最後整理成一份多工作表的 Excel 交給主管。

| 素材檔案 | 內容 |
| --- | --- |
| `POS匯出_信義店_2024-03.csv` | POS 匯出的日報：有註解行、千分位逗號、`--` 缺值、最後一列是合計 |
| `各店月報_2024.xlsx` | 5 個工作表（每店一張），前兩列是標題 |
| `外送平台訂單_2024-03.json` | 外送平台的巢狀 JSON：一筆訂單裡有多個品項 |
| `會員資料庫.db` | SQLite 資料庫，內有「會員」資料表（800 位） |
| `全年交易紀錄_2024.csv` | 10 萬筆交易紀錄（用來練習分批讀取） |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | POS 匯出的「髒」CSV——讀檔時就順便清乾淨 |
| ⬜ 任務 2 | 每家店一個工作表的 Excel——一次讀完再合併 |
| ⬜ 任務 3 | 外送平台的巢狀 JSON——攤平成表格 |
| ⬜ 任務 4 | IT 部門的會員資料庫——用 SQL 撈資料 |
| ⬜ 任務 5 | 10 萬筆大檔——分批讀取不卡頓 |
| ⬜ 任務 6 | 整理成一份多工作表 Excel 交給主管 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    'POS匯出_信義店_2024-03.csv',
    '各店月報_2024.xlsx',
    '外送平台訂單_2024-03.json',
    '會員資料庫.db',
    '全年交易紀錄_2024.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['POS匯出_信義店_2024-03.csv', '各店月報_2024.xlsx', '外送平台訂單_2024-03.json', '會員資料庫.db', '全年交易紀錄_2024.csv']


## 任務 1：POS 匯出的「髒」CSV——讀檔時就順便清乾淨

**先用文字方式看看原始檔長什麼樣子**：

In [2]:
print(open('POS匯出_信義店_2024-03.csv', encoding='utf-8-sig').read()[:600])

# 茶語時光 POS 系統匯出報表
# 門市：台北信義店　期間：2024-03-01 ~ 2024-03-31
# 匯出時間：2024-04-01 09:12:33
日期,來客數,杯數,營收,現金,電子支付,備註
2024-03-01,"273","399","22,125","5,863","16,262",
2024-03-02,"155","238","13,625","3,950","9,675",
2024-03-03,"151","236","13,610","5,371","8,239",
2024-03-04,"253","384","22,252","8,368","13,884",
2024-03-05,"208","307","16,801","6,466","10,335",
2024-03-06,"193","292","15,966","4,663","11,303",
2024-03-07,"240","395","21,426","7,020","14,406",
2024-03-08,--,--,--,--,--,系統當機未記錄
2024-03-09,"183","290","16,863","6,156","10,707",
2024-03-10,"135","208","11,034","3,995","7,039",
2024-03-11,"


**問題**：前 3 行是說明、數字有千分位逗號、`--` 代表當機沒記錄、最後一列是「合計」。直接讀會一團亂，我們用參數一次處理：

In [3]:
import pandas as pd

pos = pd.read_csv('POS匯出_信義店_2024-03.csv',
                  encoding='utf-8-sig',     # 檔案開頭有 BOM，用 utf-8-sig 讀
                  comment='#',              # # 開頭的說明行略過
                  thousands=',',            # "28,560" → 28560
                  na_values=['--'],         # -- 視為缺值
                  skipfooter=1,             # 最後一列「合計」不要
                  engine='python',          # 使用 skipfooter 需要 python 引擎
                  parse_dates=['日期'])      # 日期欄轉成時間型態
display(pos.dtypes)
print('\n缺資料的日子：\n', pos[pos['營收'].isnull()][['日期', '備註']])
print('\n3 月營收合計（不含當機日）：', f"{pos['營收'].sum():,.0f}")

日期      datetime64[us]
來客數            float64
杯數             float64
營收             float64
現金             float64
電子支付           float64
備註                 str
dtype: object


缺資料的日子：
            日期       備註
7  2024-03-08  系統當機未記錄
18 2024-03-19  系統當機未記錄

3 月營收合計（不含當機日）： 551,832


> 🤖 **換你問 AI**：「這是 CSV 的前 8 行（貼上），請用 pandas read_csv 讀取，跳過說明行、處理千分位、把 -- 當成缺值、不要讀最後的合計列。」

## 任務 2：每家店一個工作表的 Excel——一次讀完再合併

**白話說明**：`sheet_name=None` 會一次讀進「所有工作表」，得到一個「工作表名稱 → 表格」的字典；`header=2` 表示第 3 列才是欄位名稱。

In [4]:
sheets = pd.read_excel('各店月報_2024.xlsx', sheet_name=None, header=2)   # 所有工作表、第 3 列是標題
print('工作表：', list(sheets.keys()))

yearly = pd.concat(sheets, names=['門市', '列'])                            # 把 5 張表上下合併，工作表名稱變成「門市」欄
yearly = yearly.reset_index(level='門市').reset_index(drop=True)
display(yearly.head())

summary = yearly.pivot_table(index='月份', columns='門市', values='營收', sort=False)   # 月份 × 門市 的營收總表
display((summary / 10000).round(1))

工作表： ['台北信義店', '台北西門店', '台中逢甲店', '台南成大店', '高雄巨蛋店']


,門市,月份,營收,杯數
0,台北信義店,1月,501875,8886
1,台北信義店,2月,478108,8553
2,台北信義店,3月,590278,10507
3,台北信義店,4月,639347,11313
4,台北信義店,5月,749307,13030


門市,台北信義店,台北西門店,台中逢甲店,台南成大店,高雄巨蛋店
月份,,,,,
1月,50.2,55.7,68.1,49.6,57.7
2月,47.8,53.0,64.0,47.0,54.0
3月,59.0,67.9,77.7,53.1,65.7
4月,63.9,69.3,79.0,59.1,61.8
5月,74.9,81.4,89.4,60.3,70.9
6月,80.0,92.8,99.0,68.9,82.5
7月,78.9,86.6,91.9,63.9,74.3
8月,80.7,92.7,99.7,68.0,76.7
9月,71.3,80.7,87.8,61.9,75.4


## 任務 3：外送平台的巢狀 JSON——攤平成表格

**白話說明**：一筆訂單裡有好幾個品項（清單包清單）。`pd.json_normalize` 可以「攤平」：每個品項一列，並把訂單編號、門市等外層資訊帶下來。

In [5]:
import json

with open('外送平台訂單_2024-03.json', encoding='utf-8') as f:
    raw = json.load(f)
print('平台：', raw['平台'], '｜抽成：', raw['抽成比例'], '｜訂單數：', len(raw['訂單']))

items = pd.json_normalize(raw['訂單'],
                          record_path='品項',                                  # 要攤開的清單
                          meta=['訂單編號', '下單時間', ['門市', '名稱'], '外送員評分'])   # 一起帶下來的外層欄位
items['小計'] = items['數量'] * items['單價']
display(items.head())

by_store = items.groupby('門市.名稱')['小計'].sum()
print('\n各店外送營收：\n', by_store)
print('\n扣掉 30% 平台抽成，實拿：', f"{by_store.sum() * (1 - raw['抽成比例']):,.0f}", '元')

平台： FoodRun 外送 ｜抽成： 0.3 ｜訂單數： 360


,品項,數量,單價,訂單編號,下單時間,門市.名稱,外送員評分,小計
0,多多綠,3,50,FR00001,2024-03-28 12:37,台北西門店,5,150
1,四季春青茶,1,35,FR00001,2024-03-28 12:37,台北西門店,5,35
2,四季春青茶,3,35,FR00002,2024-03-22 10:03,台北西門店,4,105
3,黑糖珍珠鮮奶,1,75,FR00002,2024-03-22 10:03,台北西門店,4,75
4,黑糖珍珠鮮奶,3,75,FR00002,2024-03-22 10:03,台北西門店,4,225



各店外送營收：
 門市.名稱
台中逢甲店    17675
台北信義店    18525
台北西門店    19010
高雄巨蛋店    21915
Name: 小計, dtype: int64

扣掉 30% 平台抽成，實拿： 53,988 元


> 💡 **看懂結果**：外送營收看起來不少，但平台要抽 30%——這就是為什麼很多飲料店的外送價格比店內貴。
>
> 🤖 **換你問 AI**：「這是外送平台 JSON 的前 30 行（貼上），請用 json_normalize 把每個品項攤成一列，並帶上訂單編號與門市名稱。」

## 任務 4：IT 部門的會員資料庫——用 SQL 撈資料

In [6]:
import sqlite3

con = sqlite3.connect('會員資料庫.db')                                       # 連線到資料庫檔案
display(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))  # 資料庫裡有哪些資料表

new_2024 = pd.read_sql("""
    SELECT 常去門市, COUNT(*) AS 新會員數, ROUND(AVG(累積點數), 1) AS 平均點數
    FROM 會員
    WHERE 加入日期 >= '2024-01-01'
    GROUP BY 常去門市
    ORDER BY 新會員數 DESC
""", con)                                                                      # 2024 年加入的新會員，依門市統計
con.close()
display(new_2024)

,name
0,會員


,常去門市,新會員數,平均點數
0,台北西門店,71,300.9
1,台南成大店,66,284.7
2,高雄巨蛋店,63,253.4
3,台中逢甲店,54,278.8
4,台北信義店,47,319.1


## 任務 5：10 萬筆大檔——分批讀取不卡頓

In [7]:
display(pd.read_csv('全年交易紀錄_2024.csv', nrows=5))                         # 先偷看前 5 筆

total = pd.Series(dtype=float)
for chunk in pd.read_csv('全年交易紀錄_2024.csv', usecols=['付款方式', '金額'], chunksize=20_000):   # 每次只讀 2 萬筆、2 個欄位
    total = total.add(chunk.groupby('付款方式')['金額'].sum(), fill_value=0)      # 逐批累加
share = (total / total.sum() * 100).round(1).sort_values(ascending=False)
print('\n各付款方式金額占比 %：\n', share)

,交易編號,交易時間,門市,金額,付款方式
0,X0000001,2024-01-01 10:07:23,台南成大店,57,街口支付
1,X0000002,2024-01-01 10:07:40,台南成大店,91,LINE Pay
2,X0000003,2024-01-01 10:09:34,高雄巨蛋店,74,悠遊卡
3,X0000004,2024-01-01 10:11:51,台北信義店,300,悠遊卡
4,X0000005,2024-01-01 10:15:59,台南成大店,372,信用卡



各付款方式金額占比 %：
 付款方式
現金           32.0
LINE Pay     21.8
悠遊卡          14.0
信用卡          12.1
Apple Pay    10.1
街口支付         10.0
dtype: float64


> 💡 **看懂結果**：電子支付（LINE Pay、悠遊卡、街口、信用卡、Apple Pay）加起來已經超過現金——櫃台是不是該加裝感應設備、推出電子支付優惠？

## 任務 6：整理成一份多工作表 Excel 交給主管

In [8]:
with pd.ExcelWriter('三月結帳報告.xlsx') as writer:                           # 一個 Excel 檔、多個工作表
    pos.to_excel(writer, sheet_name='信義店3月日報', index=False)
    (summary / 10000).round(1).to_excel(writer, sheet_name='各店月營收(萬)')
    by_store.to_frame('外送營收').to_excel(writer, sheet_name='外送營收')
    new_2024.to_excel(writer, sheet_name='2024新會員', index=False)
    share.to_frame('占比%').to_excel(writer, sheet_name='付款方式')
print('✅ 已輸出 三月結帳報告.xlsx：', list(pd.read_excel('三月結帳報告.xlsx', sheet_name=None).keys()))

items.to_csv('外送品項明細.csv', index=False, encoding='utf-8-sig')            # utf-8-sig：用 Excel 開也不會亂碼

✅ 已輸出 三月結帳報告.xlsx： ['信義店3月日報', '各店月營收(萬)', '外送營收', '2024新會員', '付款方式']


---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [9]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：只讀「台南成大店」這一個工作表，找出杯數最多的月份

<details><summary>💡 卡住了？點我看提示</summary>

`sheet_name=` 填工作表名稱；這份月報的標題在第 3 列，所以 `header=2`（從 0 開始數）。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
tn = pd.read_excel('各店月報_2024.xlsx', sheet_name=___, header=2)
best_month = tn.loc[tn['杯數'].idxmax(), ___]        # 杯數最多那一列的「月份」
display(tn)
print('杯數最多的月份：', best_month)

In [ ]:
# @title ✅ 執行這格，檢查練習 1
_tn = pd.read_excel('各店月報_2024.xlsx', sheet_name='台南成大店', header=2)
check('best_month', _tn.loc[_tn['杯數'].idxmax(), '月份'])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
tn = pd.read_excel('各店月報_2024.xlsx', sheet_name='台南成大店', header=2)
best_month = tn.loc[tn['杯數'].idxmax(), '月份']
display(tn)
print('杯數最多的月份：', best_month)
```

</details>

### ✍️ 練習 2：用 SQL 撈出「累積點數前 10 名」的會員

<details><summary>💡 卡住了？點我看提示</summary>

SQL 的排序是 `ORDER BY 欄位 DESC`（由大到小），只取前幾筆是 `LIMIT 數字`。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
con = sqlite3.connect('會員資料庫.db')
top10 = pd.read_sql('SELECT 姓名, 常去門市, 累積點數 FROM 會員 ORDER BY 累積點數 ___ LIMIT ___', con)
con.close()
display(top10)

In [ ]:
# @title ✅ 執行這格，檢查練習 2
_con = sqlite3.connect('會員資料庫.db')
check('top10', pd.read_sql('SELECT 姓名, 常去門市, 累積點數 FROM 會員 ORDER BY 累積點數 DESC LIMIT 10', _con), ordered=True)
_con.close()

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
con = sqlite3.connect('會員資料庫.db')
top10 = pd.read_sql('SELECT 姓名, 常去門市, 累積點數 FROM 會員 ORDER BY 累積點數 DESC LIMIT 10', con)
con.close()
display(top10)
```

</details>

### ✍️ 練習 3：檔案太大一次讀不完：分批讀取，算出每家門市的交易筆數

<details><summary>💡 卡住了？點我看提示</summary>

`chunksize=20_000` 會讓 `read_csv` 一次給你 2 萬列；每一批用 `value_counts()` 數完，再用 `add(..., fill_value=0)` 累加起來。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
store_count = pd.Series(dtype=float)
for chunk in pd.read_csv('全年交易紀錄_2024.csv', usecols=['門市'], chunksize=___):
    store_count = store_count.add(chunk['門市'].___(), fill_value=0)
store_count = store_count.astype(int).sort_values(ascending=False)
display(store_count)

In [ ]:
# @title ✅ 執行這格，檢查練習 3
check('store_count', pd.read_csv('全年交易紀錄_2024.csv', usecols=['門市'])['門市'].value_counts())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
store_count = pd.Series(dtype=float)
for chunk in pd.read_csv('全年交易紀錄_2024.csv', usecols=['門市'], chunksize=20_000):
    store_count = store_count.add(chunk['門市'].value_counts(), fill_value=0)
store_count = store_count.astype(int).sort_values(ascending=False)
display(store_count)
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：POS 匯出的「髒」CSV——讀檔時就順便清乾淨
- ✅ 任務 2：每家店一個工作表的 Excel——一次讀完再合併
- ✅ 任務 3：外送平台的巢狀 JSON——攤平成表格
- ✅ 任務 4：IT 部門的會員資料庫——用 SQL 撈資料
- ✅ 任務 5：10 萬筆大檔——分批讀取不卡頓
- ✅ 任務 6：整理成一份多工作表 Excel 交給主管

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。